In [1]:
import importlib, env
importlib.reload(env)
from env import CloudClusterEnv, STEPS_PER_WEEK

import json
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Normal

# ---- ActorCritic (needed to load the trained agent) ----
class ActorCritic(nn.Module):
    def __init__(self, state_dim=32, action_dim=1):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, 256), nn.Tanh(),
            nn.Linear(256, 256),       nn.Tanh(),
        )
        self.actor_mean = nn.Linear(256, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))
        self.critic = nn.Linear(256, 1)
    def forward(self, state):
        x = self.shared(state)
        return self.actor_mean(x), self.critic(x)

stats = json.load(open('trace_params.json'))['stats']

# load the best trained agent
net = ActorCritic()
net.load_state_dict(torch.load('ppo_sla-focused.pth'))
net.eval()
print("Agent loaded, ready to generate decision log.")

Setup complete. Steps per week: 672
CloudClusterEnv defined.
Setup complete. Steps per week: 672
CloudClusterEnv defined.
Agent loaded, ready to generate decision log.


In [2]:
# run the trained agent through one week, capturing a rich decision log
env_log = CloudClusterEnv(stats, enable_surges=False, enable_hints=False, seed=123)
obs, _ = env_log.reset()
obs_t = torch.tensor(obs, dtype=torch.float32)

decision_log = []
prev_vms = env_log.active_vms

for t in range(STEPS_PER_WEEK):
    # capture the situation BEFORE acting (the state the agent sees)
    day, hour = env_log._current_day_hour()
    avg_cpu = env_log.history[-1][0]
    avg_mem = env_log.history[-1][2]
    queue_before = len(env_log.queue)
    vms_before = env_log.active_vms

    # agent decides (deterministic: use the mean)
    with torch.no_grad():
        mean, value = net.forward(obs_t.unsqueeze(0))
    action = mean.squeeze(0).numpy()

    # step
    obs, reward, done, tr, info = env_log.step(action)
    obs_t = torch.tensor(obs, dtype=torch.float32)

    vms_after = info['active_vms']
    delta = vms_after - vms_before
    if delta > 0:   decision = f"scaled UP by {delta}"
    elif delta < 0: decision = f"scaled DOWN by {abs(delta)}"
    else:           decision = "held steady"

    decision_log.append({
        'step': t,
        'day': day,
        'hour': hour,
        'cpu_load': round(float(avg_cpu), 3),
        'mem_load': round(float(avg_mem), 3),
        'queue_before': queue_before,
        'vms_before': vms_before,
        'vms_after': vms_after,
        'decision': decision,
        'breaches': info['breaches'],
        'cost': round(info['cost'], 3),
        'utilisation': round(info['utilisation'], 3),
    })
    if done:
        break

print(f"Captured {len(decision_log)} decisions.")
print("\nSample entries:")
for entry in decision_log[:3]:
    print(entry)

# save it
with open('decision_log.json', 'w') as f:
    json.dump(decision_log, f, indent=2)
print("\nSaved decision_log.json")

Captured 672 decisions.

Sample entries:
{'step': 0, 'day': 0, 'hour': 0, 'cpu_load': 0.0, 'mem_load': 0.0, 'queue_before': 0, 'vms_before': 4, 'vms_after': 2, 'decision': 'scaled DOWN by 2', 'breaches': 0, 'cost': 0.1, 'utilisation': 0.995}
{'step': 1, 'day': 0, 'hour': 0, 'cpu_load': 0.995, 'mem_load': 0.901, 'queue_before': 336, 'vms_before': 2, 'vms_after': 11, 'decision': 'scaled UP by 9', 'breaches': 0, 'cost': 0.55, 'utilisation': 0.999}
{'step': 2, 'day': 0, 'hour': 0, 'cpu_load': 0.999, 'mem_load': 0.965, 'queue_before': 465, 'vms_before': 11, 'vms_after': 20, 'decision': 'scaled UP by 9', 'breaches': 0, 'cost': 1.0, 'utilisation': 0.996}

Saved decision_log.json


In [3]:
def entry_to_text(e):
    """Convert one log entry into a readable sentence for embedding/retrieval."""
    return (
        f"On day {e['day']}, hour {e['hour']} (step {e['step']}): "
        f"CPU load was {e['cpu_load']:.0%}, memory load {e['mem_load']:.0%}, "
        f"with {e['queue_before']} jobs waiting in the queue. "
        f"The agent was running {e['vms_before']} VMs and {e['decision']}, "
        f"resulting in {e['vms_after']} VMs. "
        f"This step had {e['breaches']} SLA breaches, "
        f"cost {e['cost']:.2f}, and {e['utilisation']:.0%} utilisation."
    )

# convert all entries to text documents
documents = [entry_to_text(e) for e in decision_log]

print(f"Created {len(documents)} text documents.\n")
print("Example documents:")
for doc in documents[:3]:
    print("•", doc)
    print()

Created 672 text documents.

Example documents:
• On day 0, hour 0 (step 0): CPU load was 0%, memory load 0%, with 0 jobs waiting in the queue. The agent was running 4 VMs and scaled DOWN by 2, resulting in 2 VMs. This step had 0 SLA breaches, cost 0.10, and 100% utilisation.

• On day 0, hour 0 (step 1): CPU load was 100%, memory load 90%, with 336 jobs waiting in the queue. The agent was running 2 VMs and scaled UP by 9, resulting in 11 VMs. This step had 0 SLA breaches, cost 0.55, and 100% utilisation.

• On day 0, hour 0 (step 2): CPU load was 100%, memory load 96%, with 465 jobs waiting in the queue. The agent was running 11 VMs and scaled UP by 9, resulting in 20 VMs. This step had 0 SLA breaches, cost 1.00, and 100% utilisation.



In [4]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# load the embedding model (downloads ~80MB the first time)
print("Loading embedding model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# embed all 672 documents into vectors
print("Embedding documents...")
doc_embeddings = embed_model.encode(documents, show_progress_bar=True)
print(f"Embeddings shape: {doc_embeddings.shape}")   # (672, 384)

# build a FAISS index for fast similarity search
dim = doc_embeddings.shape[1]          # 384-dim vectors
index = faiss.IndexFlatL2(dim)         # L2 = Euclidean distance
index.add(np.array(doc_embeddings).astype('float32'))
print(f"FAISS index built with {index.ntotal} documents.")

# quick test: search for a question
test_question = "why did you scale up when load was high?"
q_vec = embed_model.encode([test_question]).astype('float32')
distances, indices = index.search(q_vec, k=3)   # top 3 matches

print(f"\nTop 3 documents for: '{test_question}'\n")
for rank, idx in enumerate(indices[0]):
    print(f"{rank+1}. {documents[idx]}\n")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding documents...


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Embeddings shape: (672, 384)
FAISS index built with 672 documents.

Top 3 documents for: 'why did you scale up when load was high?'

1. On day 1, hour 10 (step 137): CPU load was 99%, memory load 78%, with 98 jobs waiting in the queue. The agent was running 14 VMs and scaled UP by 1, resulting in 15 VMs. This step had 0 SLA breaches, cost 0.75, and 99% utilisation.

2. On day 3, hour 0 (step 289): CPU load was 100%, memory load 80%, with 220 jobs waiting in the queue. The agent was running 5 VMs and scaled UP by 5, resulting in 10 VMs. This step had 0 SLA breaches, cost 0.50, and 100% utilisation.

3. On day 1, hour 9 (step 133): CPU load was 61%, memory load 45%, with 0 jobs waiting in the queue. The agent was running 20 VMs and scaled DOWN by 10, resulting in 10 VMs. This step had 0 SLA breaches, cost 0.50, and 82% utilisation.



In [5]:
import ollama

def answer_question(question, k=5):
    """RAG: retrieve relevant log entries, then have Mistral answer from them."""
    # 1. RETRIEVE: find the k most relevant documents
    q_vec = embed_model.encode([question]).astype('float32')
    distances, indices = index.search(q_vec, k=k)
    retrieved = [documents[i] for i in indices[0]]

    # 2. AUGMENT: build a prompt with the retrieved facts as context
    context = "\n".join(f"- {doc}" for doc in retrieved)
    prompt = f"""You are an assistant that explains an AI cloud-scaling agent's decisions.
Answer the question using ONLY the decision records provided below.
Do not invent information. If the records don't contain the answer, say so.

Decision records:
{context}

Question: {question}

Answer in plain English, referencing specific numbers from the records:"""

    # 3. GENERATE: ask Mistral
    response = ollama.chat(model='mistral', messages=[
        {'role': 'user', 'content': prompt}
    ])
    return response['message']['content'], retrieved

# test it
question = "Why did the agent scale up on day 3?"
answer, sources = answer_question(question)

print("QUESTION:", question)
print("\nANSWER:")
print(answer)
print("\n--- based on these retrieved records ---")
for src in sources:
    print("•", src)

QUESTION: Why did the agent scale up on day 3?

ANSWER:
 The agent scaled up on day 3 because at step 302, the CPU load was 100%, memory load was 68%, and there were 68 jobs waiting in the queue. At that time, the agent was running only 2 VMs. To manage this high workload, the agent increased the number of VMs from 2 to 4, which resulted in a scaling up action.

--- based on these retrieved records ---
• On day 3, hour 3 (step 300): CPU load was 100%, memory load 74%, with 51 jobs waiting in the queue. The agent was running 5 VMs and scaled DOWN by 1, resulting in 4 VMs. This step had 0 SLA breaches, cost 0.20, and 99% utilisation.
• On day 3, hour 3 (step 301): CPU load was 99%, memory load 64%, with 30 jobs waiting in the queue. The agent was running 4 VMs and scaled DOWN by 2, resulting in 2 VMs. This step had 0 SLA breaches, cost 0.10, and 100% utilisation.
• On day 3, hour 3 (step 302): CPU load was 100%, memory load 68%, with 68 jobs waiting in the queue. The agent was running 2 

In [6]:
import gradio as gr

def chat_fn(message, history):
    answer, sources = answer_question(message)
    # append the sources so the user can see what it's grounded in
    src_text = "\n\n*Based on records:*\n" + "\n".join(f"- {s}" for s in sources[:3])
    return answer + src_text

demo = gr.ChatInterface(
    fn=chat_fn,
    title="Cloud Scaling Agent — Explainability Chatbot",
    description="Ask why the RL agent made its scaling decisions. "
                "Answers are grounded in the agent's actual decision log.",
    examples=[
        "Why did the agent scale up on day 3?",
        "When did the agent use the most VMs?",
        "Were there any SLA breaches during the week?",
        "What did the agent do during quiet periods?",
    ],
)

demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
